# Data setup

In [ ]:
from pathlib import Path

import numpy as np

import polars as pl
import polars.selectors as cs

import matplotlib.pyplot as plt


from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes import configure_mpl

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
SURVEY_COLS = [
    "participant_id",
    # "participant_type",
    "wave",
    # "start_date",
]

BELIEF_COLS = [
    "cc4_world",
    "cc4_wealthUS",
    "cc4_poorUS",
    "cc4_comm",
    "cc5_world",
    "cc5_wealthUS",
    "cc5_poorUS",
    "cc5_comm",
]

TRANSFORMS = [
    pl.col("cc1").replace({1: 2, 99: 1}),  # Move "yes" to 2, "don't know" to 1
    pl.col(r"^cc4_(world|wealthUS|poorUS|comm)$").replace(
        {1: 0, 2: 1, 99: 2}
    ),  # Shift "not at all", "only a little" down; insert "don't know" between "only a little" and "a moderate amount"
    pl.col(r"^cc5_(world|wealthUS|poorUS|comm)$").replace({1: 0, 2: 1, 99: 2}),
]

QUESTION_COLS = BELIEF_COLS

ALL_COLS = SURVEY_COLS + QUESTION_COLS

In [ ]:
config = Config(_env_file="../.env")

dataset = (
    Dataset.load(config)
    .filter_columns(ALL_COLS)
    .filter_at_least_one_resp(QUESTION_COLS)
    .impute_viterbi(QUESTION_COLS)
)
dataset_std = dataset.standardise(cs.exclude(*SURVEY_COLS))

resp = dataset.response.collect()
resp_std = dataset_std.response.collect()

# Define clustering algo

In [ ]:
from enum import StrEnum

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

import numpy.typing as npt

In [ ]:
class LinkageMethod(StrEnum):
    SINGLE = "single"
    COMPLETE = "complete"
    WARD = "ward"
    AVERAGE = "average"

In [ ]:
def cut_hclust(y: npt.NDArray[float | int], method: LinkageMethod = "single"):
    """Hierarchical clustering with pre-defined subclusters.

    Input is an M by N matrix with M observations in N dimensions.

    The dimensions are taken as pre-defined subclusters, such that the
    hierarchical clustering estimates structure between variables, rather
    than between observations (as in the `scipy.cluster.hierarchy` methods).

    An initial distance matrix is calculated based on the specified method,
    giving the distances between subclusters (variables; dimensions).
    """

    M, N = y.shape

    dists = np.full((N, N), fill_value=np.inf, dtype=np.float64)
    dists[np.diag_indices_from(dists)] = 0

    match method:
        case LinkageMethod.SINGLE:
            for i in range(N):
                for k in range(i):
                    min_dist = np.min(np.abs(y[:, i] - y[:, k]))
                    dists[i, k] = dists[k, i] = min_dist
        case LinkageMethod.COMPLETE:
            for i in range(N):
                for k in range(i):
                    max_dist = np.max(np.abs(y[:, i] - y[:, k]))
                    dists[i, k] = dists[k, i] = max_dist
        case LinkageMethod.AVERAGE:
            for i in range(N):
                for k in range(i):
                    avg_dist = np.mean(np.abs(y[:, i] - y[:, k]))
                    dists[i, k] = dists[k, i] = avg_dist
        case _:
            raise NotImplementedError

    linkage_mat = linkage(squareform(dists), method=method)

    return linkage_mat

In [ ]:
linkage_mat = cut_hclust(
    resp_std.select(*QUESTION_COLS).to_numpy(),
)

In [ ]:
linkage_mat

In [ ]:
fig, ax = plt.subplots(figsize=(3, 5))
dendrogram(linkage_mat, labels=QUESTION_COLS, orientation="right", ax=ax)

for spine in ax.spines.values():
    spine.set_visible(False)

# Scikit-learn `FeatureAgglomeration`

In [ ]:
from sklearn.cluster import FeatureAgglomeration

In [ ]:
def features_linkage(
    X: npt.NDArray[float | int], method: LinkageMethod = LinkageMethod.SINGLE
) -> npt.NDArray[np.float64]:
    """Runs scikit-learn feature agglomeration, returning linkage matrix.

    X has shape M by N with M observations in N dimensions.
    """
    M, N = X.shape

    fa = FeatureAgglomeration(
        compute_distances=True, compute_full_tree=True, linkage=method
    )
    fa.fit(X)

    Z = np.empty((N - 1, 4), dtype=np.float64)
    for i in range(N - 1):
        merged_clusters = fa.children_[i]
        merge_dist = fa.distances_[i]

        left, right = merged_clusters
        left_n_obs = 1 if left < N else Z[left - N, 3]
        right_n_obs = 1 if right < N else Z[right - N, 3]
        cluster_n_obs = left_n_obs + right_n_obs

        Z[i, [0, 1]] = merged_clusters.astype(np.float64)
        Z[i, 2] = merge_dist
        Z[i, 3] = np.float64(cluster_n_obs)

    return Z

In [ ]:
X = resp_std.select(*QUESTION_COLS).to_numpy()
linkage_mat = features_linkage(X, method=LinkageMethod.COMPLETE)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 5))
dendrogram(linkage_mat, labels=QUESTION_COLS, orientation="right", ax=ax)

for spine in ax.spines.values():
    spine.set_visible(False)

In [ ]:
fa = FeatureAgglomeration(compute_distances=True, compute_full_tree=True)
fa.fit(X)

In [ ]:
fa.distances_

In [ ]:
fa.children_